# Semana 24 · ¿Puedo predecir dónde estará Marte dentro de un año?

El proyecto final de la fase 2: simular el Sol y seis planetas, comprobar la simulación con las oposiciones de Marte y usarla para predecir.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 45-60 minutos.

## 1. Por qué importa el método: Euler contra Verlet

Simulamos solo la Tierra alrededor de un Sol fijo durante cien años, con un paso de un día, con los dos métodos. Unidades: ua, años y masas solares, con $G = 4\pi^2$ como en las semanas anteriores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt_

def acel_sol(p, G=4 * np.pi**2):
    return -G * p / np.linalg.norm(p)**3

def tierra_sola(metodo, años=100, dt=1/365.25):
    p, v = np.array([1.0, 0.0]), np.array([0.0, 2 * np.pi])     # órbita circular de 1 ua
    E = []
    a = acel_sol(p)
    for _ in range(int(años / dt)):
        if metodo == "euler":
            p, v = p + v * dt, v + acel_sol(p) * dt
        else:                                                    # Verlet
            v = v + 0.5 * dt * a; p = p + dt * v; a = acel_sol(p); v = v + 0.5 * dt * a
        E.append(0.5 * v @ v - 4 * np.pi**2 / np.linalg.norm(p))
    return np.array(E), p

for metodo in ["euler", "verlet"]:
    E, p_final = tierra_sola(metodo)
    print(f"{metodo:7s}: error relativo de la energía al final = {abs(E[-1] / E[0] - 1):.2e}   distancia final al Sol = {np.linalg.norm(p_final):.4f} ua")

Con Euler, en un siglo la Tierra ha ganado tanta energía que se ha ido a más de 4 ua. Con Verlet, la energía se mantiene y la distancia sigue en 1 ua. Usaremos Verlet.

## 2. Los elementos orbitales del JPL

Para cada planeta, la tabla del JPL da los elementos en la época J2000 (1 de enero de 2000, 12:00 de tiempo universal) y **cuánto cambian por siglo**, porque las perturbaciones de los demás planetas los hacen derivar lentamente (semana 23). Con las dos cosas se pueden calcular los elementos para cualquier fecha cercana. Usamos cuatro elementos (las órbitas en un plano) y la masa:

- $a$ (ua), $e$, la longitud media $L$ y la longitud del perihelio $\varpi$ (en grados), en J2000;
- sus variaciones por siglo;
- la masa en masas solares.

Son valores de la tabla de posiciones aproximadas del JPL (enlace en la página de la semana), escritos de memoria; la simulación los pondrá a prueba. Para la Tierra, el JPL da los elementos del centro de masas Tierra-Luna, y así los usamos.

Una precisión más: en lugar de $G = 4\pi^2$ usamos $G = k^2$ con la **constante de Gauss**, $k = 0{,}01720209895$, pasada a años de 365,25 días. $4\pi^2$ supone que un cuerpo a 1 ua tarda exactamente 365,25 días, y en realidad tarda 365,2569; la diferencia es mínima, pero en una simulación de décadas se nota.

In [ ]:
G = (0.01720209895 * 365.25) ** 2

elementos = {  #  a (ua)       e         L (°)          ϖ (°)        masa       da/siglo     de/siglo      dL/siglo (°)     dϖ/siglo (°)
    "Mercurio": (0.38709927, 0.20563593, 252.25032350,  77.45779628, 1.660e-7,  0.00000037,  0.00001906, 149472.67411175,  0.16047689),
    "Venus":    (0.72333566, 0.00677672, 181.97909950, 131.60246718, 2.448e-6,  0.00000390, -0.00004107,  58517.81538729,  0.00268329),
    "Tierra":   (1.00000261, 0.01671123, 100.46457166, 102.93768193, 3.040e-6,  0.00000562, -0.00004392,  35999.37244981,  0.32327364),
    "Marte":    (1.52371034, 0.09339410,  -4.55343205, -23.94362959, 3.227e-7,  0.00001847,  0.00007882,  19140.30268499,  0.44441088),
    "Júpiter":  (5.20288700, 0.04838624,  34.39644051,  14.72847983, 9.548e-4, -0.00011607, -0.00013253,   3034.74612775,  0.21252668),
    "Saturno":  (9.53667594, 0.05386179,  49.95424423,  92.59887831, 2.859e-4, -0.00125060, -0.00050991,   1222.49362201, -0.41897216),
}
nombres = ["Sol"] + list(elementos)
J2000 = dt_.datetime(2000, 1, 1, 12)

## 3. De los elementos a posiciones y velocidades

Para cada planeta, en la fecha de inicio:

1. Elementos en esa fecha: valor en J2000 más la variación por siglo multiplicada por los siglos transcurridos.
2. Anomalía media $M = L - \varpi$ y ecuación de Kepler, $M = E - e\sin E$ (semana 14).
3. Posición en el plano de la órbita, con el perihelio sobre el eje $x$: $(a(\cos E - e),\ a\sqrt{1-e^2}\sin E)$.
4. Velocidad: derivando la posición respecto al tiempo sale $\dfrac{\sqrt{G(1+m)a}}{r}\,(-\sin E,\ \sqrt{1-e^2}\cos E)$, con $r = a(1 - e\cos E)$.
5. Girar las dos con $R(\varpi)$ para que el perihelio apunte en su dirección real (semana 19).

Al final pasamos todo al sistema del **centro de masas** (semana 20).

In [ ]:
def resolver_kepler(M, e):
    E = M
    for _ in range(60):
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    return E

def R(ang_grados):
    t = np.radians(ang_grados)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

def estado_inicial(fecha, elementos=elementos):
    siglos = (fecha - J2000).total_seconds() / 86400 / 36525
    masas, pos, vel = [1.0], [np.zeros(2)], [np.zeros(2)]
    for a, e, L, w, m, da, de, dL, dw in elementos.values():
        a, e, L, w = a + da * siglos, e + de * siglos, L + dL * siglos, w + dw * siglos
        E = resolver_kepler(np.radians((L - w) % 360), e)
        r = a * (1 - e * np.cos(E))
        p_orb = np.array([a * (np.cos(E) - e), a * np.sqrt(1 - e**2) * np.sin(E)])
        v_orb = np.sqrt(G * (1 + m) * a) / r * np.array([-np.sin(E), np.sqrt(1 - e**2) * np.cos(E)])
        pos.append(R(w) @ p_orb); vel.append(R(w) @ v_orb); masas.append(m)
    m = np.array(masas); pos = np.array(pos); vel = np.array(vel)
    pos -= (m[:, None] * pos).sum(axis=0) / m.sum()
    vel -= (m[:, None] * vel).sum(axis=0) / m.sum()
    return m, pos, vel

## 4. La simulación de N cuerpos

Cada cuerpo siente la atracción de todos los demás. Con NumPy se calculan todas las parejas a la vez: `d[i, j]` es el vector que va del cuerpo $i$ al $j$. El paso es de un día.

In [ ]:
def aceleraciones(pos, m):
    d = pos[None, :, :] - pos[:, None, :]                       # d[i, j] = pos[j] - pos[i]
    r = np.linalg.norm(d, axis=2) + np.eye(len(m))              # el +eye evita dividir por 0 en i = j
    return G * np.sum(m[None, :, None] * d / r[:, :, None]**3, axis=1)

def simular(fecha_inicio, dias, elementos=elementos):
    m, pos, vel = estado_inicial(fecha_inicio, elementos)
    dt = 1 / 365.25
    hist = np.zeros((dias + 1, len(m), 2)); hist[0] = pos
    a = aceleraciones(pos, m)
    for k in range(dias):
        vel += 0.5 * dt * a; pos += dt * vel; a = aceleraciones(pos, m); vel += 0.5 * dt * a
        hist[k + 1] = pos
    return {"inicio": fecha_inicio, "hist": hist}

iS, iT, iM = nombres.index("Sol"), nombres.index("Tierra"), nombres.index("Marte")

def longitud(v):
    return np.degrees(np.arctan2(v[..., 1], v[..., 0])) % 360

def oposiciones(sim):
    h = sim["hist"]
    dif = (longitud(h[:, iM] - h[:, iS]) - longitud(h[:, iT] - h[:, iS]) + 180) % 360 - 180
    res = []
    for k in np.where((dif[:-1] > 0) & (dif[1:] <= 0))[0]:
        fraccion = dif[k] / (dif[k] - dif[k + 1])                # interpolación lineal entre dos días
        res.append(sim["inicio"] + dt_.timedelta(days=k + float(fraccion)))
    return res

## 5. Validación a largo plazo: desde el año 2000

Empezamos en J2000 y simulamos 28 años y medio. Marte está en **oposición** cuando su longitud heliocéntrica coincide con la de la Tierra; buscamos esos momentos. Tarda unos segundos.

In [ ]:
largo = simular(J2000, int(28.5 * 365.25))
for f in oposiciones(largo):
    print("Oposición de Marte (simulación desde 2000):", f.strftime("%Y-%m-%d %H:%M"), "UT")

h = largo["hist"]
plt.figure(figsize=(6, 6))
for i, n in enumerate(nombres[1:6], start=1):
    plt.plot(h[-800:, i, 0], h[-800:, i, 1], lw=0.8, label=n)
plt.plot(0, 0, "o", color="orange")
plt.gca().set_aspect("equal"); plt.legend(fontsize=8); plt.title("Los últimos 800 días simulados")
plt.show()

Para comparar, las fechas reales: 28 de agosto de 2003, 27 de julio de 2018, 13 de octubre de 2020, 8 de diciembre de 2022, 16 de enero de 2025 y 19 de febrero de 2027 (esta, hacia las 16:00 UT). La simulación acierta las primeras, pero se va retrasando y, al cabo de veinte años, llega con algo más de un día de retraso.

¿Por qué? Los elementos del JPL son **elementos medios**: describen la órbita promedio, sin las pequeñas oscilaciones que producen las perturbaciones. Para empezar una simulación de N cuerpos harían falta los **elementos osculadores** (del latín *osculari*, «besar»: la elipse que «besa» la trayectoria real en ese instante), que no son exactamente iguales. Un error mínimo en el semieje mayor cambia un poco el periodo, y ese error se acumula vuelta tras vuelta. Es una lección general: las simulaciones largas necesitan condiciones iniciales muy precisas.

## 6. Una predicción mejor: empezar hoy

La solución práctica es la que usan los astrónomos: empezar la simulación con datos lo más recientes posible. Calculamos los elementos medios para el 6 de noviembre de 2026 con sus variaciones por siglo, y simulamos solo 400 días. El error de partida es el mismo, pero no tiene tiempo de acumularse.

In [ ]:
HOY = dt_.datetime(2026, 11, 6)
corto = simular(HOY, 400)
for f in oposiciones(corto):
    print("Oposición de Marte (simulación desde hoy):", f.strftime("%Y-%m-%d %H:%M"), "UT    (real: 2027-02-19, hacia las 15:44 UT)")

Ahora la oposición de 2027 sale a pocos minutos de la real. Con esta simulación hacemos las predicciones.

## 7. Dónde está Marte en el cielo

Para cualquier fecha simulada calculamos:

- la **longitud eclíptica geocéntrica** $\lambda$: el ángulo del vector Tierra → Marte;
- la **ascensión recta y la declinación**: giramos la dirección eclíptica con la oblicuidad, como en la semana 19 (con latitud eclíptica 0, porque nuestro modelo es plano);
- la **elongación**: el ángulo entre las direcciones Tierra → Sol y Tierra → Marte;
- la distancia Tierra-Marte.

In [ ]:
EPS = np.radians(23.4393)     # oblicuidad de la eclíptica en J2000

def marte_en_el_cielo(sim, fecha):
    k = int(round((fecha - sim["inicio"]).total_seconds() / 86400))
    tierra, marte, sol = sim["hist"][k, iT], sim["hist"][k, iM], sim["hist"][k, iS]
    d = marte - tierra
    lam = longitud(d)
    x, y = np.cos(np.radians(lam)), np.sin(np.radians(lam))      # dirección eclíptica (β = 0)
    y_eq, z_eq = y * np.cos(EPS), y * np.sin(EPS)                # girar −ε alrededor del eje x
    ar = np.degrees(np.arctan2(y_eq, x)) % 360 / 15              # en horas
    dec = np.degrees(np.arcsin(z_eq))
    s = sol - tierra
    elong = np.degrees(np.arccos(s @ d / (np.linalg.norm(s) * np.linalg.norm(d))))
    return lam, ar, dec, elong, np.linalg.norm(d)

for fecha in [HOY, dt_.datetime(2027, 2, 19, 16), dt_.datetime(2027, 11, 6)]:
    lam, ar, dec, elong, dist = marte_en_el_cielo(corto, fecha)
    print(f"{fecha:%Y-%m-%d %H:%M}: λ = {lam:6.2f}°   AR = {int(ar)}h {(ar % 1) * 60:04.1f}m   Dec = {dec:+6.2f}°   elongación = {elong:5.1f}°   distancia = {dist:.2f} ua")

Hoy, Marte está a unos 80° del Sol: sale hacia la medianoche y se ve alto antes del amanecer. En la oposición está a 180°, visible toda la noche, a 0,68 ua. Dentro de un año estará a poco más de 30° del Sol, bajo en el horizonte del oeste después de la puesta de Sol, y más de tres veces más lejos.

**Ejercicio.** Compara las tres predicciones con Stellarium (coordenadas eclípticas J2000 de Marte en esas fechas). La declinación será la que más se aparte: nuestro modelo pone a Marte exactamente sobre la eclíptica, y en realidad puede estar hasta un par de grados por encima o por debajo, porque su órbita está inclinada. Compara también la predicción para dentro de un año con la de la simulación larga (`largo` en lugar de `corto`): ¿cuánto se diferencian?

In [ ]:
# Tu código aquí

In [ ]:
# Solución
fecha = dt_.datetime(2027, 11, 6)
l_corto = marte_en_el_cielo(corto, fecha)[0]
l_largo = marte_en_el_cielo(largo, fecha)[0]
print(f"Simulación desde hoy: λ = {l_corto:.2f}°   desde 2000: λ = {l_largo:.2f}°   diferencia: {l_largo - l_corto:+.2f}°")

## 8. El camino de Marte en el próximo año

Dibujamos la longitud geocéntrica de Marte durante 400 días. Vuelve a aparecer el retroceso de la semana 13, ahora con órbitas elípticas y todos los planetas tirando unos de otros.

In [ ]:
dias = np.arange(0, 400)
lams = np.array([marte_en_el_cielo(corto, HOY + dt_.timedelta(days=int(d)))[0] for d in dias])
plt.figure(figsize=(10, 3.5))
plt.plot(dias, np.unwrap(lams, period=360))
plt.axvline((dt_.datetime(2027, 2, 19) - HOY).days, color="r", lw=0.8, label="oposición")
plt.xlabel("días desde el 6 de noviembre de 2026"); plt.ylabel("longitud eclíptica geocéntrica (°)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 9. ¿Cuánto importa Júpiter?

Repetimos la simulación desde hoy **sin Júpiter** (con masa casi nula) y comparamos la posición de Marte vista desde la Tierra.

In [ ]:
sin_jupiter = dict(elementos)
fila = list(sin_jupiter["Júpiter"]); fila[4] = 1e-12
sin_jupiter["Júpiter"] = tuple(fila)
corto_sin = simular(HOY, 400, sin_jupiter)

for fecha in [dt_.datetime(2027, 2, 19), dt_.datetime(2027, 11, 6)]:
    dif = marte_en_el_cielo(corto, fecha)[0] - marte_en_el_cielo(corto_sin, fecha)[0]
    print(f"{fecha.date()}: diferencia en la longitud de Marte con y sin Júpiter = {60 * dif:+.2f}′")

En un año, Júpiter cambia la posición de Marte en el cielo alrededor de un minuto de arco: invisible a simple vista, pero mucho más que la precisión de las efemérides profesionales. Y el efecto crece con el tiempo.

## 10. Experimenta

1. Duplica el paso de tiempo y repite la predicción desde hoy (tendrás que añadir un argumento `dt` a `simular`). ¿Cuánto cambia la fecha de la oposición? ¿Y con 10 días?
2. Usa la simulación para predecir la siguiente oposición de Marte después de 2027. Compárala con lo que dé Stellarium.
3. Con la simulación larga, calcula la distancia entre la Tierra y Marte en cada oposición desde 2000. ¿Por qué cambia tanto? Pista: semana 14.

In [ ]:
# Tu experimento aquí